In [1]:
import wmpaws
import pandas as pd

In [6]:
wikis = ['pawiki', 'hiwiki', 'aswiki', 'mlwiki', 'bhwiki', 'mrwiki', 'knwiki',
         'tawiki', 'sawiki', 'tewiki', 'bnwiki', 'guwiki', 'satwiki', 'awawiki']

In [9]:
first_rights_query = """
SELECT 
    log_title AS user_name,
    MIN(log_timestamp) AS first_rights_change
FROM logging
WHERE log_type = 'rights'
GROUP BY log_title
"""

In [10]:
registration_query = """
SELECT 
    user_name,
    user_registration
FROM user
"""

In [11]:
rights_time_list = []

for wiki in wikis:
    try:
        first_rights_df = wmpaws.run_sql(first_rights_query, wiki)
        reg_df = wmpaws.run_sql(registration_query, wiki)
        
        merged = first_rights_df.merge(reg_df, on='user_name', how='left')
        merged['wiki'] = wiki
        rights_time_list.append(merged)
    except Exception as e:
        print(f"Failed on {wiki}: {e}")

rights_time_df = pd.concat(rights_time_list, ignore_index=True)
rights_time_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2097 entries, 0 to 2096
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   user_name            2097 non-null   str  
 1   first_rights_change  2097 non-null   str  
 2   user_registration    1556 non-null   str  
 3   wiki                 2097 non-null   str  
dtypes: str(4)
memory usage: 152.6 KB


In [12]:
# Keep only rows where we have both dates
clean_df = rights_time_df.dropna(subset=['user_registration', 'first_rights_change']).copy()

# Convert MediaWiki's text timestamps into real datetime objects
clean_df['user_registration'] = pd.to_datetime(clean_df['user_registration'], format='%Y%m%d%H%M%S')
clean_df['first_rights_change'] = pd.to_datetime(clean_df['first_rights_change'], format='%Y%m%d%H%M%S')

# Time between registration and first rights grant
clean_df['days_to_first_right'] = (clean_df['first_rights_change'] - clean_df['user_registration']).dt.days

clean_df[['user_name', 'wiki', 'user_registration', 'first_rights_change', 'days_to_first_right']].head(10)

,user_name,wiki,user_registration,first_rights_change,days_to_first_right
0,AlleborgoBot,pawiki,2007-10-26 17:21:21,2007-12-21 20:31:22,56
1,AntiCompositeNumber,pawiki,2020-09-02 21:33:18,2022-03-03 03:56:51,546
2,AramilFeraxa,pawiki,2023-06-12 12:02:39,2025-03-17 09:00:32,643
3,Babanwalia,pawiki,2011-02-17 08:38:17,2014-07-09 04:15:31,1237
5,Base,pawiki,2012-12-06 14:13:51,2023-02-10 16:38:49,3718
7,BotMultichill,pawiki,2007-08-26 14:49:21,2007-10-14 20:03:02,49
8,Bsadowski1,pawiki,2009-07-22 20:18:02,2011-06-02 04:42:21,679
10,DragonBot,pawiki,2007-10-19 17:34:40,2007-11-26 01:02:31,37
11,Drini,pawiki,2007-10-09 01:59:32,2008-04-02 18:59:01,176
12,EPIC,pawiki,2023-11-21 07:07:29,2024-04-12 14:38:54,143


In [13]:
print(f"Total rights-change records: {len(rights_time_df)}")
print(f"With valid registration date: {len(clean_df)}")
print(f"Coverage rate: {len(clean_df) / len(rights_time_df):.1%}")

print(clean_df['days_to_first_right'].describe())

Total rights-change records: 2097
With valid registration date: 1556
Coverage rate: 74.2%
count    1556.000000
mean      685.807841
std       997.716789
min     -5408.000000
25%        43.750000
50%       271.500000
75%       944.250000
max      6658.000000
Name: days_to_first_right, dtype: float64


In [14]:
# Keep only realistic values: rights granted on or after registration
valid_df = clean_df[clean_df['days_to_first_right'] >= 0].copy()

print(f"Rows before filtering negatives: {len(clean_df)}")
print(f"Rows after filtering negatives: {len(valid_df)}")
print(f"Negative (likely bad data) rows removed: {len(clean_df) - len(valid_df)}")

print(valid_df['days_to_first_right'].describe())

Rows before filtering negatives: 1556
Rows after filtering negatives: 1554
Negative (likely bad data) rows removed: 2
count    1554.000000
mean      691.056628
std       984.906976
min         0.000000
25%        44.000000
50%       272.500000
75%       944.750000
max      6658.000000
Name: days_to_first_right, dtype: float64


In [15]:
valid_df.groupby('wiki')['days_to_first_right'].describe()

,count,mean,std,min,25%,50%,75%,max
wiki,,,,,,,,
aswiki,40.0,533.200000,908.725966,0.0,20.00,113.0,627.75,4517.0
awawiki,8.0,843.000000,721.390918,7.0,387.75,606.5,1398.00,2103.0
bhwiki,28.0,340.250000,672.644401,0.0,18.75,72.0,239.25,2668.0
bnwiki,287.0,569.376307,862.779889,0.0,18.50,196.0,770.00,5493.0
guwiki,55.0,509.000000,934.703181,0.0,56.50,116.0,515.00,5020.0
hiwiki,254.0,684.881890,992.213449,0.0,55.50,274.5,912.75,6176.0
knwiki,44.0,834.409091,1537.200904,0.0,24.00,99.0,663.50,6658.0
mlwiki,266.0,746.210526,939.998953,0.0,52.50,421.0,1135.25,6117.0
mrwiki,101.0,633.772277,1069.286396,0.0,28.00,131.0,655.00,4820.0
